In [362]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

In [363]:
df = pd.read_csv("campusvolt_energy_data.csv")

In [364]:
df.head()

,timestamp,meterId,buildingId,buildingName,voltage,current,power,energy,frequency,powerFactor,temperature,occupancyLevel,weatherCondition,dayType,workingHours,scenario,status,simulationTime,simulationSpeed
0,2025-01-01 00:00:00,M001,B001,Academic Block,232.30,53.68,19.37,19.37,49.962,0.891,19.82,6.59,Sunny,Weekday,False,Normal,Normal,2025-01-01 00:00:00,1
1,2025-01-01 01:00:00,M001,B001,Academic Block,226.96,44.72,15.56,34.93,50.025,0.866,20.73,0.00,Sunny,Weekday,False,Normal,Normal,2025-01-01 01:00:00,1
2,2025-01-01 02:00:00,M001,B001,Academic Block,230.33,33.41,11.96,46.89,49.908,0.889,20.94,7.99,Sunny,Weekday,False,Normal,Normal,2025-01-01 02:00:00,1
3,2025-01-01 03:00:00,M001,B001,Academic Block,232.47,31.70,11.61,58.50,49.902,0.883,19.98,0.00,Cloudy,Weekday,False,Normal,Normal,2025-01-01 03:00:00,1
4,2025-01-01 04:00:00,M001,B001,Academic Block,229.10,38.65,13.48,71.98,49.882,0.873,21.42,9.41,Cloudy,Weekday,False,Normal,Normal,2025-01-01 04:00:00,1


In [365]:
print(df.shape)

(43800, 19)


In [366]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 43800 entries, 0 to 43799
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   timestamp         43800 non-null  str    
 1   meterId           43800 non-null  str    
 2   buildingId        43800 non-null  str    
 3   buildingName      43800 non-null  str    
 4   voltage           43800 non-null  float64
 5   current           43800 non-null  float64
 6   power             43800 non-null  float64
 7   energy            43800 non-null  float64
 8   frequency         43800 non-null  float64
 9   powerFactor       43800 non-null  float64
 10  temperature       43800 non-null  float64
 11  occupancyLevel    43800 non-null  float64
 12  weatherCondition  43800 non-null  str    
 13  dayType           43800 non-null  str    
 14  workingHours      43800 non-null  bool   
 15  scenario          43800 non-null  str    
 16  status            43800 non-null  str    
 17  simu

In [367]:
df.isnull().sum()

timestamp           0
meterId             0
buildingId          0
buildingName        0
voltage             0
current             0
power               0
energy              0
frequency           0
powerFactor         0
temperature         0
occupancyLevel      0
weatherCondition    0
dayType             0
workingHours        0
scenario            0
status              0
simulationTime      0
simulationSpeed     0
dtype: int64

In [368]:
df.describe()

,voltage,current,power,energy,frequency,powerFactor,temperature,occupancyLevel,simulationSpeed
count,43800.000000,43800.000000,43800.000000,43800.000000,43800.000000,43800.000000,43800.000000,43800.000000,43800.0
mean,230.016157,125.717601,44.689157,198519.377843,49.999974,0.887742,30.451116,29.890713,1.0
std,3.001855,91.497930,32.844700,134845.936081,0.079923,0.019233,6.070816,28.796576,0.0
min,217.760000,13.120000,5.000000,5.680000,49.696000,0.813000,13.110000,0.000000,1.0
25%,227.990000,54.500000,19.140000,80004.067500,49.946000,0.874000,25.860000,5.570000,1.0
50%,230.030000,99.230000,34.660000,189833.425000,50.000000,0.887000,30.460000,15.290000,1.0
75%,232.020000,177.090000,63.510000,286277.195000,50.054000,0.900000,35.080000,56.090000,1.0
max,241.830000,665.480000,234.460000,512660.460000,50.376000,0.967000,46.670000,100.000000,1.0


In [369]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 43800 entries, 0 to 43799
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   timestamp         43800 non-null  datetime64[us]
 1   meterId           43800 non-null  str           
 2   buildingId        43800 non-null  str           
 3   buildingName      43800 non-null  str           
 4   voltage           43800 non-null  float64       
 5   current           43800 non-null  float64       
 6   power             43800 non-null  float64       
 7   energy            43800 non-null  float64       
 8   frequency         43800 non-null  float64       
 9   powerFactor       43800 non-null  float64       
 10  temperature       43800 non-null  float64       
 11  occupancyLevel    43800 non-null  float64       
 12  weatherCondition  43800 non-null  str           
 13  dayType           43800 non-null  str           
 14  workingHours      43800 non-null 

In [370]:
df["hour"] = df["timestamp"].dt.hour
df["day"] = df["timestamp"].dt.day
df["dayOfWeek"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month

In [371]:
df[["timestamp", "hour", "day", "dayOfWeek", "month"]].head()

,timestamp,hour,day,dayOfWeek,month
0,2025-01-01 00:00:00,0,1,2,1
1,2025-01-01 01:00:00,1,1,2,1
2,2025-01-01 02:00:00,2,1,2,1
3,2025-01-01 03:00:00,3,1,2,1
4,2025-01-01 04:00:00,4,1,2,1


In [372]:
df["powerLag1"] = df.groupby("meterId")["power"].shift(1)

In [373]:
df["powerLag24"] = df.groupby("meterId")["power"].shift(24)

In [374]:
df = df.dropna()
df.shape

(43680, 25)

In [375]:
features = [
   "hour",
    "day",
    "dayOfWeek",
    "month",
    "powerLag1",
    "powerLag24",
    "temperature",
    "occupancyLevel"
]

In [376]:
target = "power"

In [377]:
X = df[features]
y = df[target]

In [378]:
df = df.sort_values(["meterId", "timestamp"]).reset_index(drop=True)

In [379]:
df.head()

,timestamp,meterId,buildingId,buildingName,voltage,current,power,energy,frequency,powerFactor,...,scenario,status,simulationTime,simulationSpeed,hour,day,dayOfWeek,month,powerLag1,powerLag24
0,2025-01-02 00:00:00,M001,B001,Academic Block,232.28,40.09,14.30,1120.27,49.938,0.880,...,Normal,Normal,2025-01-02 00:00:00,1,0,2,3,1,18.55,19.37
1,2025-01-02 01:00:00,M001,B001,Academic Block,228.58,38.87,13.78,1134.05,50.087,0.881,...,Normal,Normal,2025-01-02 01:00:00,1,1,2,3,1,14.30,15.56
2,2025-01-02 02:00:00,M001,B001,Academic Block,228.05,34.20,12.39,1146.44,50.172,0.890,...,Normal,Normal,2025-01-02 02:00:00,1,2,2,3,1,13.78,11.96
3,2025-01-02 03:00:00,M001,B001,Academic Block,231.51,21.71,7.55,1153.99,50.069,0.863,...,Normal,Normal,2025-01-02 03:00:00,1,3,2,3,1,12.39,11.61
4,2025-01-02 04:00:00,M001,B001,Academic Block,232.75,22.93,8.47,1162.46,50.170,0.895,...,Normal,Normal,2025-01-02 04:00:00,1,4,2,3,1,7.55,13.48


In [380]:
splitIndex = int(len(df) * 0.8)

XTrain = X.iloc[:splitIndex]
XTest = X.iloc[splitIndex:]

yTrain = y.iloc[:splitIndex]
yTest = y.iloc[splitIndex:]

In [381]:
model = LinearRegression()

In [382]:
model.fit(XTrain, yTrain)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](8,)","[-0.14, 0.01,-1.51,..., 0.18, 0.83, 0.4 ]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](8,)","['hour','day','dayOfWeek',...,'powerLag24','temperature','occupancyLevel']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,-14.98
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,8
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(8)


In [383]:
yPred = model.predict(XTest)

In [384]:
yTest

35064    12.88
35065    11.11
35066     5.32
35067     5.00
35068     5.00
         ...  
43795    14.64
43796    17.62
43797     7.02
43798    12.33
43799    13.77
Name: power, Length: 8736, dtype: float64

In [385]:
mae = mean_absolute_error(yTest, yPred)
rmse = np.sqrt(mean_squared_error(yTest, yPred))
r2 = r2_score(yTest, yPred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R² Score:", r2)

MAE: 5.341297973627237
RMSE: 7.098351497943229
R² Score: 0.9166388893742325


In [386]:
model = RandomForestRegressor()

In [387]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    max_depth=20,
    min_samples_split=10
)

In [388]:
model.fit(XTrain,yTrain)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of

In [389]:
ypredict = model.predict(XTest)

In [390]:
mae = mean_absolute_error(yTest, ypredict)
rmse = np.sqrt(mean_squared_error(yTest, ypredict))
r2 = r2_score(yTest, ypredict)

print("MAE:", mae)
print("RMSE:", rmse)
print("R² Score:", r2)

MAE: 4.713402378029932
RMSE: 7.041076716765606
R² Score: 0.9179787011051462


In [391]:
yPredict = model.predict(XTrain)

In [392]:
mae = mean_absolute_error(yTrain, yPredict)
rmse = np.sqrt(mean_squared_error(yTrain, yPredict))
r2 = r2_score(yTrain, yPredict)

print("MAE:", mae)
print("RMSE:", rmse)
print("R² Score:", r2)

MAE: 2.2075538850545855
RMSE: 3.386510791729202
R² Score: 0.9900868265538283
